# SMAP L3 Soil Moisture Increase and Saturation Filter

This notebook reads SMAP L3 soil moisture NetCDF data and finds dates where at least a cutoff fraction of stations show increases. It also detects saturation periods for the same stations and trims those dates to end 5 days before the decline begins.

In [97]:
import numpy as np
import pandas as pd
import netCDF4 as nc
from pathlib import Path
from datetime import datetime, timedelta
import warnings
import os

warnings.filterwarnings('ignore')
os.chdir('/master_storage6/ishan/my_data/SSM-irrigation-data/SMAP')

print('Libraries imported successfully!')

Libraries imported successfully!


In [98]:
# Configuration
DATA_FILE = Path('/master_storage6/ishan/my_data/SSM-irrigation-data/SMAP/SPL3SMP_Ludhiana_ROI.nc')

START_DATE = '2015-01-01'
STOP_DATE = '2025-12-31'

SM_SOURCE = 'BOTH'  # Options: 'AM', 'PM', 'BOTH'
FILL_VALUE = -9999.0

# Station cutoff fraction (e.g., 0.8 means 80% of stations)
CUTOFF_FRACTION = 0.75

PRE_EVENT_DAYS = 1
POST_EVENT_DAYS = 2

# Thresholds for increase/saturation/decline checks
INCREASE_THRESHOLD = 0.05
SATURATION_TOLERANCE = 0.0
DECLINE_THRESHOLD = 0.05

# Saturation trimming: stop 5 days before decline starts
SATURATION_LEAD_DAYS = 5

OUTPUT_DIR = Path('/master_storage6/ishan/my_data/SSM-irrigation-data/SMAP')
INCREASE_OUT = OUTPUT_DIR / 'smap_increase_dates.txt'
SATURATION_OUT = OUTPUT_DIR / 'smap_saturation_dates_trimmed.txt'

print('Configuration:')
print(f'  Data file: {DATA_FILE}')
print(f'  Date range: {START_DATE} to {STOP_DATE}')
print(f'  SM source: {SM_SOURCE}')
print(f'  Fill value excluded: {FILL_VALUE}')
print(f'  Cutoff fraction: {CUTOFF_FRACTION}')
print(f'  Increase threshold: {INCREASE_THRESHOLD}')
print(f'  Saturation tolerance: {SATURATION_TOLERANCE}')
print(f'  Decline threshold: {DECLINE_THRESHOLD}')
print(f'  Saturation lead days: {SATURATION_LEAD_DAYS}')
print(f'  Output files: {INCREASE_OUT}, {SATURATION_OUT}')

Configuration:
  Data file: /master_storage6/ishan/my_data/SSM-irrigation-data/SMAP/SPL3SMP_Ludhiana_ROI.nc
  Date range: 2015-01-01 to 2025-12-31
  SM source: BOTH
  Fill value excluded: -9999.0
  Cutoff fraction: 0.75
  Increase threshold: 0.05
  Saturation tolerance: 0.0
  Decline threshold: 0.05
  Saturation lead days: 5
  Output files: /master_storage6/ishan/my_data/SSM-irrigation-data/SMAP/smap_increase_dates.txt, /master_storage6/ishan/my_data/SSM-irrigation-data/SMAP/smap_saturation_dates_trimmed.txt


In [99]:
# Load NetCDF data
if not DATA_FILE.exists():
    raise FileNotFoundError(f'File not found: {DATA_FILE}')

ds = nc.Dataset(DATA_FILE, 'r')

time_j2000 = ds.variables['time'][:]
epoch = datetime(2000, 1, 1)
time_dt = np.array([epoch + timedelta(seconds=float(t)) for t in time_j2000])

latitude = ds.variables['latitude'][:]
longitude = ds.variables['longitude'][:]
date_str = ds.variables['date'][:]
soil_moisture_am = ds.variables['soil_moisture_am'][:]
soil_moisture_pm = ds.variables['soil_moisture_pm'][:]

ds.close()

df = pd.DataFrame({
    'time': time_dt,
    'date_str': date_str,
    'latitude': latitude,
    'longitude': longitude,
    'soil_moisture_am': soil_moisture_am,
    'soil_moisture_pm': soil_moisture_pm
})

# Exclude fill values
df = df[(df['latitude'] != FILL_VALUE) & (df['longitude'] != FILL_VALUE)]
df = df[(df['soil_moisture_am'] != FILL_VALUE) | (df['soil_moisture_pm'] != FILL_VALUE)]

if SM_SOURCE == 'AM':
    df['soil_moisture'] = df['soil_moisture_am'].replace(FILL_VALUE, np.nan)
elif SM_SOURCE == 'PM':
    df['soil_moisture'] = df['soil_moisture_pm'].replace(FILL_VALUE, np.nan)
else:
    sm_am = df['soil_moisture_am'].replace(FILL_VALUE, np.nan)
    sm_pm = df['soil_moisture_pm'].replace(FILL_VALUE, np.nan)
    df['soil_moisture'] = np.nanmean([sm_am, sm_pm], axis=0)

df = df.dropna(subset=['soil_moisture'])

df = df[(df['time'] >= START_DATE) & (df['time'] <= STOP_DATE)]

print(f'Valid observations after filtering: {len(df):,}')
print(f"Date range: {df['time'].min()} to {df['time'].max()}")

Valid observations after filtering: 19,925
Date range: 2015-04-03 00:00:00 to 2025-12-31 00:00:00


In [100]:
# Aggregate daily soil moisture per location
unique_locations = df[['latitude', 'longitude']].drop_duplicates().reset_index(drop=True)
unique_locations['location_id'] = range(len(unique_locations))

df_with_locations = df.merge(
    unique_locations[['latitude', 'longitude', 'location_id']],
    on=['latitude', 'longitude'],
    how='left'
)

df_with_locations['date'] = pd.to_datetime(df_with_locations['time']).dt.date

location_time_series = df_with_locations.groupby(['location_id', 'date']).agg({
    'soil_moisture': 'mean',
    'latitude': 'first',
    'longitude': 'first'
}).reset_index()

location_time_series.columns = ['location_id', 'date', 'sm_mean', 'latitude', 'longitude']
location_time_series['date'] = pd.to_datetime(location_time_series['date'])

# Pivot to a date x location matrix
ts = location_time_series.pivot_table(index='date', columns='location_id', values='sm_mean')
ts = ts.sort_index()

print(f'Date count: {len(ts.index)}')
print(f'Station count: {len(ts.columns)}')

Date count: 1878
Station count: 12


In [101]:
# ---------------------------------------------------------
# STEP 1: Define Saturation Thresholds per Station (for Exclusion)
# ---------------------------------------------------------
# We define saturation as being within 5% of the station's max recorded value
# We want to EXCLUDE these events now.
station_max = ts.max()
saturation_thresholds = station_max * 0.95

# ---------------------------------------------------------
# STEP 2: Vectorized Daily Classification
# ---------------------------------------------------------

# Calculate daily change (SM_t - SM_{t-1})
daily_diff = ts.diff()

# Condition A: Increasing by at least 0.02
# (User requested threshold of 0.02)
is_increasing = daily_diff >= 0.02

# Condition B: Saturated (Monsoon/Standing water signature)
# We identify these to EXCLUDE them.
is_saturated = ts >= saturation_thresholds

# Combine: Station is valid if it is Increasing AND NOT Saturated
# We look for increases that happen below saturation levels.
# ~is_saturated gives True where it is NOT saturated.
is_valid_event = is_increasing & (~is_saturated)

# ---------------------------------------------------------
# STEP 3: Apply Spatial Filter (The 70% Rule)
# ---------------------------------------------------------

# Count valid data points per day (exclude NaNs in original data)
# Note: ts.count(axis=1) counts non-NaN values.
valid_counts = ts.count(axis=1)

# Count how many stations met the criteria per day
event_counts = is_valid_event.sum(axis=1)

# Calculate fraction
# Avoid division by zero
daily_fractions = event_counts / valid_counts
daily_fractions = daily_fractions.fillna(0.0)

# Filter dates where fraction >= 0.70
calibration_dates = daily_fractions[daily_fractions >= CUTOFF_FRACTION].index

print(f"Total dates analyzed: {len(ts)}")
print(f"Dates identified for calibration (Increase > 0.02, Not Saturated): {len(calibration_dates)}")

# ---------------------------------------------------------
# STEP 4: Export
# ---------------------------------------------------------

# ---------------------------------------------------------
# STEP 5: Expand Single Dates into Full Event Windows
# ---------------------------------------------------------

# How many days of "drying" to include after a rain event?
# 4-5 days is standard for surface soil moisture (0-5cm)


#-------------------------------------------
# EVENT_WINDOW_DAYS = 2

# # Use a set to automatically handle overlaps 
# # (e.g., if it rains on Day 1 and Day 3, the windows merge naturally)
# expanded_date_set = set()

# for date in calibration_dates:
#     # For every rain date, include the next N days
#     for i in range(EVENT_WINDOW_DAYS + 1):
#         future_date = date + pd.Timedelta(days=i)
        
#         # Check boundary: Don't go past the end of your dataset
#         if future_date <= ts.index[-1]:
#             expanded_date_set.add(future_date)

# # Convert back to a sorted DatetimeIndex
# final_calibration_dates = pd.DatetimeIndex(sorted(list(expanded_date_set)))

# print(f"Original 'Peak' Dates: {len(calibration_dates)}")
# print(f"Final 'Window' Dates for Calibration: {len(final_calibration_dates)}")

# # ---------------------------------------------------------
# # STEP 6: Save the Final List
# # ---------------------------------------------------------
# final_df = pd.DataFrame({'date': final_calibration_dates})
# final_df.to_csv(OUTPUT_DIR / 'smap_calibration_windows.txt', index=False)
#---------------------------------
# ---------------------------------------------------------
# IMPROVED EVENT WINDOW EXPANSION
# ---------------------------------------------------------

# 1. Capture the "Pre-Event" baseline (t-1)
# 2. Capture the "Recession" tail (t+1, t+2, t+3)
#    - 3 days is better for fitting drainage (b) than 2 days


expanded_date_set = set()

for date in calibration_dates:
    # Range starts from -1 (Yesterday) to +3 (3 days after)
    for i in range(-PRE_EVENT_DAYS, POST_EVENT_DAYS + 1):
        target_date = date + pd.Timedelta(days=i)
        
        # Boundary Check: Ensure we don't go outside the dataset
        if target_date >= ts.index[0] and target_date <= ts.index[-1]:
            expanded_date_set.add(target_date)

# Convert to sorted Index
final_calibration_dates = pd.DatetimeIndex(sorted(list(expanded_date_set)))

print(f"Original Trigger Dates: {len(calibration_dates)}")
print(f"Final Physically Consistent Dates: {len(final_calibration_dates)}")
print(f"Percentage of Total Data: {len(final_calibration_dates)/len(ts)*100:.1f}%")

# Save
final_df = pd.DataFrame({'date': final_calibration_dates})
final_df.to_csv(OUTPUT_DIR / 'smap_calibration_optimal.txt', index=False)

print(f"Saved extended events to: {OUTPUT_DIR / 'smap_calibration_windows.txt'}")
calibration_df = pd.DataFrame({
    'date': calibration_dates,
    'fraction_stations': daily_fractions[calibration_dates].values
})

# Save
# calibration_df.to_csv(OUTPUT_DIR / 'smap_calibration_dates.txt', index=False)
# print(f"Saved to: {OUTPUT_DIR / 'smap_calibration_dates.txt'}")

# Optional: Preview the first few dates
print("\nFirst 10 Calibration Dates:")
print(calibration_df.head(10))

Total dates analyzed: 1878
Dates identified for calibration (Increase > 0.02, Not Saturated): 76
Original Trigger Dates: 76
Final Physically Consistent Dates: 295
Percentage of Total Data: 15.7%
Saved extended events to: /master_storage6/ishan/my_data/SSM-irrigation-data/SMAP/smap_calibration_windows.txt

First 10 Calibration Dates:
        date  fraction_stations
0 2015-04-08           1.000000
1 2015-04-16           1.000000
2 2015-04-29           1.000000
3 2015-06-01           0.916667
4 2015-09-21           0.833333
5 2015-12-07           0.833333
6 2015-12-12           0.750000
7 2016-02-20           1.000000
8 2016-03-06           0.916667
9 2016-03-20           0.750000


In [102]:
# # Save outputs to text files
# if not increase_df.empty:
#     increase_df = increase_df.sort_values('date')
#     increase_df.to_csv(INCREASE_OUT, sep=',', index=False)

# if not saturation_df.empty:
#     saturation_df = saturation_df.sort_values(['event_start', 'saturation_date'])
#     saturation_df.to_csv(SATURATION_OUT, sep=',', index=False)

# print(f'Increase output: {INCREASE_OUT} (rows: {len(increase_df)})')
# print(f'Saturation output: {SATURATION_OUT} (rows: {len(saturation_df)})')